#Example 9.4.1 Microservices for Machine Learning

In [ ]:
# Install necessary libraries
!pip install flask joblib google-cloud-storage

# Import required libraries
from flask import Flask, request, jsonify
import joblib
import threading
from google.cloud import storage

# Initialize Google Cloud Storage client
BUCKET_NAME = 'my-bucket-name'  # Update with your bucket name
MODEL_FILE = 'model.pkl'  # Path to your model in the GCS bucket

# Download the model from GCS to Colab
def download_model():
    client = storage.Client()
    bucket = client.bucket(BUCKET_NAME)
    blob = bucket.blob(MODEL_FILE)
    blob.download_to_filename(MODEL_FILE)
    print(f"Model downloaded from GCS: {MODEL_FILE}")

download_model()

# Load the pre-trained model
model = joblib.load(MODEL_FILE)

# Initialize the Flask app
app = Flask(__name__)

# Define a route for the prediction API
@app.route('/predict', methods=['POST'])
def predict():
    data = request.json
    features = data['features']
    prediction = model.predict([features])
    return jsonify({'prediction': prediction.tolist()})

# Function to run Flask app
def run_flask():
    app.run(host='0.0.0.0', port=8080)

# Start the Flask app in a separate thread
threading.Thread(target=run_flask).start()

# Sample prediction request to test the API
import requests
import time

# Wait for the server to start
time.sleep(3)

# Define the API endpoint
url = 'http://localhost:8080/predict'
data = {'features': [1.0, 2.0, 3.0, 4.0]}  # Replace with your actual feature data

# Send the POST request to the API
response = requests.post(url, json=data)

# Print the prediction response
print(response.json())




In [ ]:
#Example 9.4.2 serverless Computing for Machine Learning on Google Cloud
import json
import joblib
from google.cloud import storage

# Define the Google Cloud Storage bucket and model file
BUCKET_NAME = 'my-bucket-name'  # Replace with your bucket name
MODEL_FILE = 'model.pkl'  # Replace with your model file name

# Function to download the model from GCS
def download_model():
    client = storage.Client()
    bucket = client.bucket(BUCKET_NAME)
    blob = bucket.blob(MODEL_FILE)
    blob.download_to_filename('/tmp/' + MODEL_FILE)  # Save to /tmp for Cloud Functions
    print(f"Model {MODEL_FILE} downloaded from GCS")

# Load the pre-trained model from GCS
download_model()
model = joblib.load('/tmp/' + MODEL_FILE)

def predict(request):
    # Parse the JSON request
    request_json = request.get_json()
    features = request_json['features']

    # Make prediction
    prediction = model.predict([features])

    # Return the prediction as a JSON response
    return json.dumps({'prediction': prediction.tolist()})
#Deployment Command:
#gcloud functions deploy predict --runtime python39 --trigger-http --allow-unauthenticated --memory 512MB

In [ ]:
#Example 9.4.3 Containerization for Machine Learning on GCP
#Dockerfile
# Use an official Python runtime as a parent image
FROM python:3.8-slim

# Set the working directory in the container
WORKDIR /app

# Copy the current directory contents into the container at /app
COPY . /app

# Install required packages from requirements.txt
RUN pip install --no-cache-dir -r requirements.txt

# Expose port 8080 for Flask app
EXPOSE 8080

# Run app.py when the container launches
CMD ["python", "app.py"]


In [ ]:
#1. Build and Push the Docker Container to Google Container Registry (GCR)
# Authenticate with GCP (if not already authenticated)
gcloud auth login

# Set the project ID
gcloud config set project your-project-id

# Build the Docker image
docker build -t gcr.io/your-project-id/ml-model .

# Push the image to Google Container Registry (GCR)
docker push gcr.io/your-project-id/ml-model

In [ ]:
#2. Create a Google Kubernetes Engine (GKE) Cluster:
# Create a GKE cluster with 3 nodes
gcloud container clusters create ml-cluster --num-nodes=3

In [ ]:
#3. Deploy the Docker Container to the GKE Cluster:
# Deploy the container to the GKE cluster
kubectl create deployment ml-model --image=gcr.io/your-project-id/ml-model

# Expose the deployment as a service with a LoadBalancer
kubectl expose deployment ml-model --type=LoadBalancer --port 80 --target-port 8080

In [ ]:
#4. Get the External IP to Access the Service:
# Retrieve the external IP of the service
kubectl get services ml-model

#9.5 Google Cloud Services for LLMOps

In [ ]:
#Example: Training an LLM on Google Cloud AI Platform
from google.cloud import aiplatform

# Initialize AI Platform
aiplatform.init(project='your-project-id', location='us-central1')

# Define the custom training job
job = aiplatform.CustomTrainingJob(
    display_name='llm-training-job',
    script_path='train.py',  # Path to your training script
    container_uri='gcr.io/cloud-aiplatform/training/tf-cpu.2-3:latest',  # Specify the container with the environment
    requirements=['transformers', 'datasets']  # Any required packages
)

# Run the training job on AI Platform
job.run(
    replica_count=1,  # Number of replicas
    model_display_name='llm-model',  # Display name for the trained model
    args=['--epochs', '5', '--batch-size', '32']  # Arguments passed to the training script
)


In [ ]:
#Example: Using Google Cloud Functions to Deploy a Language Model
import json
import joblib
from google.cloud import storage

# Initialize the Google Cloud Storage client
storage_client = storage.Client()

# Define your bucket and model location
bucket = storage_client.get_bucket('your-bucket-name')
blob = bucket.blob('model.pkl')

# Download the model to a temporary directory in the Colab or GCP environment
blob.download_to_filename('/tmp/model.pkl')

# Load the model using joblib
model = joblib.load('/tmp/model.pkl')

# Define the prediction function
def predict(request):
    # Parse the request
    request_json = request.get_json()
    input_text = request_json.get('input_text', '')

    # Generate a response using the loaded model
    prediction = model.predict([input_text])

    # Return the prediction as a JSON response
    return json.dumps({'prediction': prediction})

In [ ]:
#Example: Storing and Accessing Model Checkpoints in Google Cloud Storage
from google.cloud import storage

# Initialize the Google Cloud Storage client
storage_client = storage.Client()

# Upload a model checkpoint to GCS
bucket = storage_client.bucket('your-bucket-name')  # Replace with your bucket name
blob = bucket.blob('models/checkpoint.pt')  # Replace with the desired path in your bucket
blob.upload_from_filename('checkpoint.pt')  # Replace with the local path of your checkpoint

# Download a model checkpoint from GCS
blob = bucket.blob('models/checkpoint.pt')  # Replace with the path in your bucket
blob.download_to_filename('checkpoint_downloaded.pt')  # Replace with your local destination file path

In [ ]:
#Example: Deploying an LLM in a Docker Container on GKE
#dockerfile
FROM python:3.8-slim

WORKDIR /app

COPY . /app

RUN pip install --no-cache-dir -r requirements.txt

EXPOSE 8080

CMD ["python", "app.py"]

#other steps
# Build and push the Docker image to Google Container Registry
docker build -t gcr.io/book-examples-2024/llm-service .
docker push gcr.io/book-examples-2024/llm-service

# Create a GKE cluster
gcloud container clusters create llm-cluster --num-nodes=3

# Deploy the container to the GKE cluster
kubectl create deployment llm-service --image=gcr.io/book-examples-2024/llm-service

# Expose the deployment as a service
kubectl expose deployment llm-service --type=LoadBalancer --port 80 --target-port 8080


In [ ]:
#Example: Analyzing Large Datasets in BigQuery for LLM Training
from google.cloud import bigquery

# Initialize the BigQuery client
client = bigquery.Client()

# Define the query to select training data
query = """
    SELECT text_data
    FROM `book-examples-2024.dataset.table`
    WHERE label = 'relevant'
"""

# Execute the query
query_job = client.query(query)
results = query_job.result()

# Process the results for training
training_data = [row.text_data for row in results]

#Example 9.7.1 Blue-Green Deployment

In [ ]:
# Deploy the Blue Environment:
gcloud run deploy ml-service-blue --image gcr.io/book-examples-2024/ml-model:v1 --platform managed --region us-central1

# Deploy the Green Environment with the New Model Version:
gcloud run deploy ml-service-green --image gcr.io/book-examples-2024/ml-model:v2 --platform managed --region us-central1

# Switch Traffic to the Green Environment:
gcloud run services update-traffic ml-service-green --to-revisions ml-service-green=50,ml-service-blue=50

# Complete the Traffic Shift:
gcloud run services update-traffic ml-service-green --to-revisions ml-service-green=100

#Example 9.7.2 Canary Deployment

In [ ]:
# Deploy the Canary Version:
kubectl create deployment ml-service-canary --image=gcr.io/book-examples-2024/ml-model:v2
kubectl expose deployment ml-service-canary --type=LoadBalancer --port 80 --target-port 8080

# Route a Small Percentage of Traffic to the Canary Version:
gcloud compute url-maps import ml-url-map --source url-map.yaml

# Sample url-map.yaml
defaultRouteAction:
  weightedBackendServices:
    - backendService: ml-service-canary-backend
      weight: 10
    - backendService: ml-service-stable-backend
      weight: 90

# Monitor Performance and Gradually Increase Traffic:
gcloud compute url-maps update ml-url-map --source url-map.yaml

#Example 9.7.3 Continuous Integration and Continuous Deployment (CI/CD)

In [ ]:
# cloudbuild.yaml for CI/CD pipeline
steps:
- name: 'gcr.io/cloud-builders/docker'
  args: ['build', '-t', 'gcr.io/book-examples-2024/ml-model:v2', '.']

- name: 'gcr.io/cloud-builders/docker'
  args: ['push', 'gcr.io/book-examples-2024/ml-model:v2']

- name: 'gcr.io/cloud-builders/kubectl'
  args: ['set', 'image', 'deployment/ml-service', 'ml-container=gcr.io/book-examples-2024/ml-model:v2']

# Trigger the CI/CD pipeline on commits to the main branch
gcloud builds triggers create github --name="ml-model-update-trigger" --repo-name="your-repo-name" --branch-pattern="^main$" --build-config="cloudbuild.yaml"


#Example 9.7.4 A/B Testing

In [ ]:
# Deploy version A and B on Google Cloud Run
gcloud run deploy ml-service-A --image gcr.io/book-examples-2024/ml-model:v1 --platform managed --region us-central1
gcloud run deploy ml-service-B --image gcr.io/book-examples-2024/ml-model:v2 --platform managed --region us-central1

# Split traffic 50/50 between version A and B
gcloud run services update-traffic ml-service-A --to-revisions ml-service-A=50,ml-service-B=50


#Example 9.7.5 Shadow Deployment

In [ ]:
# Deploy the shadow model on Google Kubernetes Engine (GKE)
kubectl create deployment ml-service-shadow --image=gcr.io/book-examples-2024/ml-model:v2
# Expose the shadow service without routing traffic
kubectl expose deployment ml-service-shadow --type=ClusterIP --port 8080

# Use Google Cloud Pub/Sub to mirror traffic to the shadow deployment
gcloud pubsub topics publish ml-traffic --message='{"input_data": "data_to_mirror"}'

# Compare predictions (using mirroring)
gcloud pubsub topics publish ml-traffic --message='{"input_data": "data_to_mirror"}'

#Example 9.7.5 Model Versioning and Rollback

In [ ]:
from google.cloud import aiplatform

# Initialize AI Platform
aiplatform.init(project='book-examples-2024', location='us-central1')

# Upload a new model version to Vertex AI Model Registry
model = aiplatform.Model.upload(
    display_name='ml-model',
    artifact_uri='gs://my-image-data-bucket-unique/models/v2',
    serving_container_image_uri='gcr.io/cloud-aiplatform/prediction/tf2-cpu.2-5:latest'
)

# Deploy version 2 of the model from Vertex AI Model Registry
!gcloud ai endpoints create --display-name="ml-model-endpoint"
!gcloud ai endpoints deploy-model ml-model-endpoint --model=projects/book-examples-2024/locations/us-central1/models/ml-model@2

# Rollback to version 1 if version 2 has issues
!gcloud ai endpoints deploy-model ml-model-endpoint --model=projects/book-examples-2024/locations/us-central1/models/ml-model@1 --traffic-split=0=100


#Example 9.8.2 Auto Scaling in Google Kubernetes Engine (GKE)

In [ ]:
#yaml file
apiVersion: apps/v1
kind: Deployment
metadata:
  name: ml-model-deployment
spec:
  replicas: 2
  selector:
    matchLabels:
      app: ml-model
  template:
    metadata:
      labels:
        app: ml-model
    spec:
      containers:
      - name: ml-model-container
        image: gcr.io/book-examples-2024/ml-model:latest
        resources:
          requests:
            cpu: "500m"
          limits:
            cpu: "1"


In [ ]:
# Create an HPA for the Deployment
kubectl autoscale deployment ml-model-deployment --cpu-percent=50 --min=2 --max=10

#Example 9.8.3 Auto Scaling in Google Compute Engine

In [ ]:
# 1 Create a Managed Instance Group
gcloud compute instance-groups managed create ml-instance-group \
    --base-instance-name ml-instance \
    --template ml-instance-template \
    --size 1 \
    --zone us-central1-a

# 2 Define an Auto Scaling Policy
gcloud compute instance-groups managed set-autoscaling ml-instance-group \
    --max-num-replicas 10 \
    --min-num-replicas 2 \
    --target-cpu-utilization 0.6 \
    --cool-down-period 90 \
    --zone us-central1-a

#Example 9.8.4 Auto Scaling in Google Cloud Run

In [ ]:
# Deploy a Cloud Run Service with Auto Scaling Configuration
gcloud run deploy ml-service \
    --image gcr.io/your-project-id/ml-model:latest \
    --platform managed \
    --region us-central1 \
    --min-instances 1 \
    --max-instances 10

#9.8.5 Monitoring and Adjusting Auto Scaling Policies

In [ ]:
# Create an Alert Policy for CPU Utilization
gcloud monitoring policies create \
    --display-name "High CPU Utilization" \
    --condition-display-name "CPU utilization above 80%" \
    --condition-threshold 80 \
    --condition-filter 'metric.type="compute.googleapis.com/instance/cpu/utilization" AND resource.type="gce_instance"' \
    --notification-channels email:your-email@example.com


#9.9 Cost Optimization Strategies

In [ ]:
#Example: Using Google Cloud’s Recommender for Rightsizing
# List recommendations for resource rightsizing
gcloud recommender recommendations list \
    --project=your-project-id \
    --location=global \
    --recommender=google.compute.instance.MachineTypeRecommender

In [ ]:
#Example: Purchasing Committed Use Contracts for Compute Engine
# Purchase a committed use contract for Compute Engine
gcloud compute commitments create \
    --project=your-project-id \
    --name=my-commitment \
    --plan=3-year \
    --region=us-central1 \
    --resources=cores=16,memory=60


In [ ]:
#Example: Deploying a Preemptible VM
# Create a preemptible VM instance on Google Compute Engine
gcloud compute instances create preemptible-instance \
    --project=your-project-id \
    --zone=us-central1-a \
    --machine-type=n1-standard-1 \
    --preemptible

In [ ]:
#Example: Moving Data to a Lower-Cost Storage Class
gsutil mv gs://your-bucket/important-data.csv gs://your-bucket-nearline/important-data.csv

In [ ]:
#Example: Implementing Auto Scaling in Google Kubernetes Engine
# Enable auto scaling for a GKE cluster
gcloud container clusters update my-cluster \
    --enable-autoscaling \
    --min-nodes=1 \
    --max-nodes=10 \
    --zone=us-central1-a

In [ ]:
#Example: Scheduling Auto Shutdown for Idle Resources
# Schedule a shutdown for an idle VM instance using a cron job
gcloud compute instances stop my-instance \
    --project=your-project-id \
    --zone=us-central1-a \
    --schedule="0 2 * * *"  # Stop instance daily at 2 AM


In [ ]:
#Example: Setting Up Budget Alerts
gcloud billing budgets create \
    --project=your-project-id \
    --display-name="My Budget" \
    --amount=1000USD \
    --threshold-rule=percentage=0.8


In [ ]:
#Example: Using VPC Peering to Reduce Networking Costs
# Create VPC peering between two networks
gcloud compute networks peerings create my-peering \
    --network=my-vpc-network-1 \
    --peer-network=my-vpc-network-2 \
    --auto-create-routes

#Example 9.10.1 Security on Google Cloud

In [ ]:
#Example: Setting Up IAM Policies
# Grant a user the role of Storage Admin on a specific bucket
gcloud projects add-iam-policy-binding your-project-id \
    --member="user:example-user@example.com" \
    --role="roles/storage.admin" \
    --condition=None \
    --resource="projects/your-project-id"


In [ ]:
#Example: Creating and Using a Customer-Managed Encryption Key (CMEK)
# Create a new key ring and key for encryption
gcloud kms keyrings create my-key-ring --location=global
gcloud kms keys create my-key --location=global --keyring=my-key-ring --purpose=encryption
# Use the key to encrypt a Cloud Storage bucket
gsutil mb -p your-project-id -l us-central1 -c standard \
    -k projects/your-project-id/locations/global/keyRings/my-key-ring/cryptoKeys/my-key \
    gs://my-secure-bucket/


In [ ]:
#Example: Configuring Firewall Rules
gcloud compute firewall-rules create allow-ssh \
    --network default \
    --allow tcp:22 \
    --source-ranges 192.168.1.0/24

#Example 9.10.2 Compliance on Google Cloud


In [ ]:
#Example: Selecting a Data Location for Cloud Storage
gsutil mb -p your-project-id -l europe-west1 gs://my-europe-bucket/


In [ ]:
Example: Enabling and Viewing Audit Logs
# Enable audit logging for a Google Cloud service
gcloud services enable logging.googleapis.com --project=your-project-id


In [ ]:
# View audit logs in Google Cloud Console
gcloud logging read "resource.type=gce_instance AND logName:activity" --limit 100
